In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from scipy import io
from tensorflow.keras.callbacks import ModelCheckpoint
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split

In [2]:
all_images = list()
all_labels = None

image_files_path = "datasets/font_images/"
image_files = os.listdir(image_files_path)
image_files = sorted(image_files, key = lambda x: int(x.split('.')[0]))

image_count = len(image_files)

for i in range(image_count):
    file = image_files[i]
    all_images.append(cv2.imread(image_files_path + file, cv2.IMREAD_GRAYSCALE))
    # if i == 0:
    #     cv2.imshow("hufs", all_images[0])
    #     cv2.waitKey(1000)

with open("datasets/font_labels.txt", "r") as f:
    lines = f.read().splitlines()

lines_int = list()

for i in lines:
    lines_int.append(ord(i) - 65)

all_labels = np.array(lines_int)
all_images = np.array(all_images)

train_images, test_images, train_labels, test_labels = train_test_split(all_images, all_labels, test_size=0.1, random_state=42, shuffle=True)

In [ ]:
print(train_images.shape, train_labels.shape, test_images.shape, test_labels.shape)

# cv2.imshow("hdcs", all_images[2])
# cv2.waitKey(10)
# print(all_labels[2])

#print(image_files)

In [4]:
model = models.Sequential([

    layers.Conv2D(32, (3,3), padding="same", use_bias="false"),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.Conv2D(32, (3,3), padding="same", use_bias="false"),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(64, (3,3), padding="same", use_bias="false"),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.20),
    layers.Dense(26, activation="softmax")
])

model.compile(optimizer="adam", loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [5]:
train_images = train_images / 255.0
test_images = test_images / 255.0

train_images = train_images.reshape(len(train_images), 28, 28, 1)
test_images = test_images.reshape(len(test_images), 28, 28, 1)

checkpoint = ModelCheckpoint(filepath="checkpoints/epoch_{epoch:02d}.keras", save_weights_only=False, save_freq="epoch")

model.fit(train_images, train_labels, epochs=15,
          validation_data=(test_images, test_labels), callbacks=[checkpoint], batch_size=24)

# 100 % val accuracy and < 0.0028 loss

Epoch 1/15
662/662 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.8250 - loss: 0.6072 - val_accuracy: 0.9938 - val_loss: 0.0532
Epoch 2/15
662/662 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.9592 - loss: 0.1287 - val_accuracy: 0.9949 - val_loss: 0.0234
Epoch 3/15
662/662 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9685 - loss: 0.0923 - val_accuracy: 0.9977 - val_loss: 0.0096
Epoch 4/15
662/662 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9733 - loss: 0.0733 - val_accuracy: 0.9989 - val_loss: 0.0057
Epoch 5/15
662/662 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9808 - loss: 0.0574 - val_accuracy: 0.9977 - val_loss: 0.0045
Epoch 6/15
662/662 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.9788 - loss: 0.0606 - val_accuracy: 0.9994 - val_loss: 0.0027
Epoch 7/15
662/662 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.9834 - loss: 0.0475 - val_accuracy: 0.9887 - val_loss: 0.0298
Epoch 8/15
662/662 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.9801 - loss: 0.0562 - 

In [ ]:
test_images = test_images / 255.0
test_images = test_images / 255.0

test_images = test_images.reshape(len(test_images), 28, 28, 1)
test_images = test_images.reshape(len(test_images), 28, 28, 1)

model = tf.keras.models.load_model("font_identifier.keras")
loss, acc = model.evaluate(test_images, test_labels, batch_size=64)
print("Accuracy: ", acc*100, " Loss: ", loss)